# Book Market Analytics – Books to Scrape
**Data Science 500 | End-to-end project notebook**

## 1. Project objective
This notebook collects book catalogue information from the Books to Scrape practice website, cleans the data, explores pricing patterns, and trains a linear regression model to estimate prices.

The required fields are book title, price in GBP, star rating, and stock availability/quantity where shown. The notebook aims to scrape the full catalogue and checks that at least 200 records were collected.

**Run all cells from top to bottom.** Internet access is required for the scraping step. Model metrics and findings are calculated from the data collected during the run.

## 2. Import libraries and set up paths
Install the packages from `requirements.txt` first if needed.

In [4]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
# If the notebook is launched from the notebooks folder, move up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "scraper.py").exists():
    # This also supports opening the notebook from another working directory.
    candidate = Path("../")
    if (candidate / "src" / "scraper.py").exists():
        PROJECT_ROOT = candidate.resolve()

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from scraper import scrape_catalogue, clean_dataset

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = PROJECT_ROOT / "reports"
for folder in (DATA_DIR, OUTPUT_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT.resolve())

ImportError: DLL load failed while importing groupby: An Application Control policy has blocked this file.

## 3. Data acquisition with BeautifulSoup
The scraper follows the catalogue's **Next** link until the final page. It reads listing pages and extracts the fields specified in the assignment. The website is intended for scraping practice; a short delay is included between catalogue pages.

In [ ]:
raw_df = scrape_catalogue()
print(f"Raw rows collected: {len(raw_df)}")
display(raw_df.head())

if len(raw_df) < 200:
    print("WARNING: fewer than 200 books were collected. Check the website response and run again.")
else:
    print("The assignment's minimum of 200 books has been met.")

## 4. Data cleaning and preparation
Cleaning steps:
- convert price, rating and stock quantity to numeric values;
- standardise text fields;
- remove duplicate listings;
- remove rows without a usable title or price;
- create a binary `stock_available` feature for analysis.

Missing stock quantity is not automatically replaced with zero because a missing quantity does not necessarily mean zero stock.

In [ ]:
books = clean_dataset(raw_df)
csv_path = DATA_DIR / "books_dataset.csv"
books.to_csv(csv_path, index=False)

print("Cleaned shape:", books.shape)
display(books.head())
print("\nMissing values:")
display(books.isna().sum().to_frame("missing_count"))
print("\nSummary statistics:")
display(books[["price_gbp", "rating", "stock_quantity"]].describe())
print(f"\nSaved dataset to: {csv_path.resolve()}")

## 5. Exploratory data analysis
The following plots investigate price distribution, average price by star rating, and the relationship between rating and price. Each chart is followed by a short interpretation based on the collected data.

In [2]:
# Plot 1: distribution of prices
plt.figure(figsize=(9, 5))
plt.hist(books["price_gbp"].dropna(), bins=25, edgecolor="black")
plt.title("Distribution of Book Prices")
plt.xlabel("Price (£ GBP)")
plt.ylabel("Number of books")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_distribution.png", dpi=150)
plt.show()

print("Interpretation: Use the histogram to identify the price range containing the most books and whether the distribution is skewed.")

NameError: name 'plt' is not defined

In [ ]:
# Plot 2: average price by star rating
rating_summary = (
    books.dropna(subset=["rating"])
         .groupby("rating", as_index=False)["price_gbp"]
         .mean()
         .sort_values("rating")
)
plt.figure(figsize=(8, 5))
plt.bar(rating_summary["rating"].astype(str), rating_summary["price_gbp"], edgecolor="black")
plt.title("Average Book Price by Star Rating")
plt.xlabel("Star rating (1–5)")
plt.ylabel("Average price (£ GBP)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "average_price_by_rating.png", dpi=150)
plt.show()
display(rating_summary)
print("Interpretation: Compare the bar heights to see whether books with higher ratings have higher average prices. This is association, not proof of causation.")

In [ ]:
# Plot 3: rating versus price
plot_data = books.dropna(subset=["rating", "price_gbp"])
plt.figure(figsize=(8, 5))
plt.scatter(plot_data["rating"], plot_data["price_gbp"], alpha=0.65)
plt.title("Book Price versus Star Rating")
plt.xlabel("Star rating (1–5)")
plt.ylabel("Price (£ GBP)")
plt.xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_vs_rating.png", dpi=150)
plt.show()

if len(plot_data) > 1:
    corr = plot_data["rating"].corr(plot_data["price_gbp"])
    print(f"Pearson correlation between rating and price: {corr:.3f}")
    print("Interpretation: Values close to +1 indicate a positive linear association; values close to -1 indicate a negative association; values near 0 indicate a weak linear association.")

### Stock availability exploration
This chart compares the number of listings marked in stock with other statuses.

In [ ]:
stock_counts = books["stock_status"].value_counts()
plt.figure(figsize=(7, 4))
plt.bar(stock_counts.index.astype(str), stock_counts.values, edgecolor="black")
plt.title("Book Listings by Stock Status")
plt.xlabel("Stock status")
plt.ylabel("Number of books")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stock_status_counts.png", dpi=150)
plt.show()
display(stock_counts.to_frame("book_count"))

## 6. Predictive modelling: linear regression
The target variable is `price_gbp`. Candidate features are rating, stock availability, stock quantity, and the rating category. Since the data may have missing ratings or stock quantities, the modelling pipeline imputes missing numeric values and one-hot encodes the categorical feature. The train/test split evaluates performance on books not used to fit the model.

Metrics:
- **MAE**: average absolute prediction error in GBP (lower is better).
- **RMSE**: gives greater weight to large errors (lower is better).
- **R²**: proportion of target variation explained by the model; it can be negative if the model performs worse than predicting the test-set mean.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_df = books.dropna(subset=["price_gbp"]).copy()
model_df["rating_category"] = model_df["rating"].astype("string").fillna("Unknown")
model_df["stock_quantity"] = model_df["stock_quantity"].astype(float)
model_df["stock_available"] = model_df["stock_available"].astype(float)

numeric_features = ["rating", "stock_available", "stock_quantity"]
categorical_features = ["rating_category"]
X = model_df[numeric_features + categorical_features]
y = model_df["price_gbp"]

if len(model_df) < 10:
    raise ValueError("Not enough records to train a meaningful model. Collect more data and rerun.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE (£)", "RMSE (£)", "R²"],
    "Value": [mae, rmse, r2]
})
display(metrics)

In [ ]:
# Compare actual and predicted prices for the held-out test set.
plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.7)
low = min(float(y_test.min()), float(np.min(predictions)))
high = max(float(y_test.max()), float(np.max(predictions)))
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Actual versus Predicted Book Prices")
plt.xlabel("Actual price (£ GBP)")
plt.ylabel("Predicted price (£ GBP)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=150)
plt.show()

print(f"MAE: £{mae:.2f}. On average, predictions differ from actual prices by about this amount.")
print(f"RMSE: £{rmse:.2f}. Larger errors have more influence on this metric.")
print(f"R²: {r2:.3f}. Interpret this together with MAE/RMSE and the size and variety of the dataset.")

## 7. Results and limitations
The model is a baseline. Price can depend on variables not included in the listing features, such as book length, publisher, format, edition, and genre. The website is a practice catalogue, so findings should not be generalised automatically to the entire publishing market. A relationship between rating and price does not establish causation.

In [ ]:
# Create a report summary using the actual values calculated above.
mean_price = books["price_gbp"].mean()
median_price = books["price_gbp"].median()
min_price = books["price_gbp"].min()
max_price = books["price_gbp"].max()
rating_means = books.dropna(subset=["rating"]).groupby("rating")["price_gbp"].mean()
if len(rating_means):
    lowest_rating_group = int(rating_means.idxmin())
    highest_rating_group = int(rating_means.idxmax())
    rating_comparison = (
        f"Rating group {lowest_rating_group} had the lowest average price "
        f"(£{rating_means.min():.2f}); rating group {highest_rating_group} had the highest "
        f"(£{rating_means.max():.2f})."
    )
else:
    rating_comparison = "There were not enough valid ratings to compare average prices by rating."

report = f'''# Book Market Analytics – Project Report

## Introduction
This project analyses the Books to Scrape practice catalogue to explore book pricing and build a baseline linear regression model for estimating prices.

## Methodology
BeautifulSoup was used to extract book listings from catalogue pages. The dataset included title, price in GBP, star rating, stock status, and stock quantity where displayed. Data preparation standardised data types, removed duplicate rows and unusable price/title entries, and created a binary stock availability feature. Visualisations were created with Matplotlib. A linear regression model was evaluated using a held-out test set.

## Dataset overview
- Cleaned books: {len(books)}
- Mean price: £{mean_price:.2f}
- Median price: £{median_price:.2f}
- Minimum price: £{min_price:.2f}
- Maximum price: £{max_price:.2f}

## Exploratory findings
{rating_comparison}
The price distribution and rating/price charts saved in the outputs folder should be reviewed to describe the most visible patterns. Correlation indicates association and does not prove causation.

## Predictive modelling
The linear regression model used rating, stock availability, stock quantity, and a rating category feature. On the held-out test set:
- MAE: £{mae:.2f}
- RMSE: £{rmse:.2f}
- R²: {r2:.3f}

MAE and RMSE are expressed in GBP. R² should be interpreted alongside the error metrics. Results may vary if the source catalogue changes.

## Limitations
The model uses a limited set of catalogue features. Important price factors such as publisher, edition, format, page count and genre may not be available in the listing-level data. Missing stock quantities are retained as missing and imputed only within the modelling pipeline. Results from this practice website should not be treated as representative of all book retailers.

## Recommendations
1. Use the price distribution to identify common price bands and unusually high-priced listings.
2. Compare price patterns across rating groups without assuming ratings cause higher or lower prices.
3. Collect richer features, such as genre, publisher, format and page count, before relying on the model for business decisions.
4. Re-run the scraping and evaluation workflow when the catalogue changes and compare the metrics across runs.

## Conclusion
The project demonstrates an end-to-end workflow covering web scraping, cleaning, visual exploration, linear regression and reporting. The model is a baseline and should be improved with additional features and further validation before operational use.
'''
report_path = REPORT_DIR / "project_report.md"
report_path.write_text(report, encoding="utf-8")
print("Report saved to:", report_path.resolve())
print(report)